# 38. Pretrained Color-Invariant Adapter 실험

`색에 속는다` 가설을 직접 겨냥하되, SegFormer-B0 pretrained 조건이 깨지지 않도록 입력 앞단 adapter만 추가합니다.

비교 원칙:

- A0: 기존 pretrained vanilla SegFormer-B0 baseline은 Chapter 2-2 산출물을 재사용합니다.
- A1: 기존 pretrained vanilla + photometric augmentation도 Chapter 2-2 산출물을 재사용합니다.
- A2: pretrained SegFormer-B0 + RGB identity-initialized grayscale/edge adapter를 새로 학습합니다.
- A3: A2에 color counterfactual consistency loss를 추가합니다.

A2/A3는 SegFormer 본체를 pretrained로 시작하고 adapter는 처음에 RGB identity로 동작합니다. 따라서 구조 변경 모델이 pretrained를 잃어서 불리해지는 문제를 피합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch3_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/3장/ch3_utils.py")) + list(Path.cwd().glob("**/ch3_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "3장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch3_utils import *

paths = find_ch3_paths()
set_korean_font()
set_seed(38)
paths

Chapter3Paths(chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), chapter2_2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장'), data_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data'), stress_ladder_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data/synthetic_metal_stress_ladder'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs/manifests'), ch2_2_runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs'))

## 38-1. Manifest와 비교군 산출물 확인

In [2]:
samples = load_ch3_base_samples()
manifests = create_ch3_probe_manifests(samples, max_per_cell=None, seed=38)
registry = discover_ch3_model_registry()

required = {
    "baseline_seed_metrics": paths.ch2_2_runs_root / "baseline_seed_matched_sample_metrics.csv",
    "strategy_summary": paths.ch2_2_runs_root / "strategy_comparison_summary.csv",
    "photometric_seed0": paths.ch2_2_runs_root / "strategy_seed_repeats" / "photometric_aug" / "seed_0" / "sample_metrics.csv",
}
missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"비교군 산출물이 부족합니다: {missing}")

for name, path in manifests.items():
    print(f"{name:32s}", path, "rows=", len(pd.read_csv(path)))
display(registry[["family", "variant", "model_seed", "checkpoint_exists"]].head(12))

train                            C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\manifests\standard\train_manifest.csv rows= 288
eval_matched                     C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\manifests\standard\eval_matched_manifest.csv rows= 240
eval_stress                      C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\manifests\standard\eval_stress_manifest.csv rows= 160
eval_matched_probe               C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\manifests\standard\eval_matched_probe_manifest.csv rows= 240
eval_color_counterfactual_probe  C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\manifests\standard\eval_color_counterfactual_probe_manifest.csv rows= 192


,family,variant,model_seed,checkpoint_exists
0,baseline,baseline_no_aug,0,True
1,baseline,baseline_no_aug,1,True
2,baseline,baseline_no_aug,2,True
3,strategy,group_balanced,0,True
4,strategy,group_balanced,1,True
5,strategy,group_balanced,2,True
6,strategy,photometric_aug,0,True
7,strategy,photometric_aug,1,True
8,strategy,photometric_aug,2,True
9,exposure_ratio,red_scratch_ratio_0p00,0,True


## 38-2. Pretrained 유지 조건 명시

In [3]:
experiment_contract = {
    "hypothesis": "red 실패는 RGB appearance shortcut 때문에 발생한다.",
    "intervention": "SegFormer pretrained 본체 앞에 RGB identity-initialized grayscale/edge adapter를 추가한다.",
    "fairness_controls": [
        "SegFormer backbone/head config는 pretrained vanilla B0와 동일하게 시작한다.",
        "adapter는 초기 forward가 RGB identity와 같도록 초기화한다.",
        "train/eval manifest, seed, epoch, batch size, lr은 2-2장 baseline과 맞춘다.",
        "from-scratch 구조 모델과 pretrained vanilla를 직접 비교하지 않는다.",
    ],
    "success_criteria": {
        "red_dice_delta_vs_baseline": ">= +0.30",
        "seen_color_dice_drop_vs_baseline": ">= -0.05",
        "red_original_vs_grayscale_gap": "<= 0.15 is ideal",
        "counterfactual_stability": "prediction flip and Dice gap should decrease",
    },
}
out_dir = paths.runs_root / "color_adapter_pretrained"
out_dir.mkdir(parents=True, exist_ok=True)
save_json(out_dir / "experiment_contract.json", experiment_contract)
print(json.dumps(experiment_contract, ensure_ascii=False, indent=2))

{
  "hypothesis": "red 실패는 RGB appearance shortcut 때문에 발생한다.",
  "intervention": "SegFormer pretrained 본체 앞에 RGB identity-initialized grayscale/edge adapter를 추가한다.",
  "fairness_controls": [
    "SegFormer backbone/head config는 pretrained vanilla B0와 동일하게 시작한다.",
    "adapter는 초기 forward가 RGB identity와 같도록 초기화한다.",
    "train/eval manifest, seed, epoch, batch size, lr은 2-2장 baseline과 맞춘다.",
    "from-scratch 구조 모델과 pretrained vanilla를 직접 비교하지 않는다."
  ],
  "success_criteria": {
    "red_dice_delta_vs_baseline": ">= +0.30",
    "seen_color_dice_drop_vs_baseline": ">= -0.05",
    "red_original_vs_grayscale_gap": "<= 0.15 is ideal",
    "counterfactual_stability": "prediction flip and Dice gap should decrease"
  }
}


## 38-3. A2/A3 학습

In [4]:
MODEL_SEEDS = [0, 1, 2]
EPOCHS = 30
BATCH_SIZE = 8
LR = 1e-3

variants = [
    {"name": "adapter_identity_ce", "consistency": False},
    {"name": "adapter_identity_consistency", "consistency": True},
]

run_root = out_dir / "runs"
for variant in variants:
    for seed in MODEL_SEEDS:
        run_dir = run_root / variant["name"] / f"seed_{seed}"
        if (run_dir / "sample_metrics.csv").exists():
            print("skip existing:", run_dir)
            continue
        print("training:", variant["name"], "seed", seed)
        train_pretrained_color_adapter_experiment(
            train_manifest=manifests["train"],
            eval_manifest=manifests["eval_matched"],
            run_dir=run_dir,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LR,
            seed=seed,
            consistency=variant["consistency"],
            consistency_weight=0.25,
            consistency_ce_weight=0.5,
        )

training: adapter_identity_ce seed 0


C:\Users\준승\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 208/208 [00:00<00:00, 15262.15it/s]


[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([6, 256, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([6])                      

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


...

epoch 26 | loss=0.0162 | ce=0.0108 | cons=0.0002


epoch 27 | loss=0.0149 | ce=0.0099 | cons=0.0001


epoch 28 | loss=0.0143 | ce=0.0093 | cons=0.0002


epoch 29 | loss=0.0160 | ce=0.0106 | cons=0.0002


epoch 30 | loss=0.0151 | ce=0.0100 | cons=0.0001


## 38-4. A0/A1/A2/A3 성능 비교

In [5]:
adapter_seed_metrics = collect_color_adapter_metrics(run_root, out_dir=out_dir)
adapter_summary = pd.read_csv(out_dir / "color_adapter_summary.csv")

baseline = pd.read_csv(paths.ch2_2_runs_root / "baseline_seed_matched_sample_metrics.csv")
baseline_rows = []
for seed, part in baseline.groupby("model_seed"):
    heldout = part[part["color_group"].isin(HELDOUT_COLORS)]
    seen = part[part["color_group"].isin(SEEN_COLORS)]
    baseline_rows.append({
        "variant": "baseline_no_aug",
        "model_seed": seed,
        "mean_dice": part["target_dice"].mean(),
        "seen_color_dice": seen["target_dice"].mean(),
        "heldout_color_dice": heldout["target_dice"].mean(),
        "red_dice": part[part["color_group"] == "red"]["target_dice"].mean(),
        "purple_dice": part[part["color_group"] == "purple"]["target_dice"].mean(),
        "worst_combo_dice": part.groupby(["color_group", "defect_type"])["target_dice"].mean().min(),
    })

photo_rows = []
for seed in MODEL_SEEDS:
    p = paths.ch2_2_runs_root / "strategy_seed_repeats" / "photometric_aug" / f"seed_{seed}" / "sample_metrics.csv"
    if not p.exists():
        continue
    part = pd.read_csv(p)
    heldout = part[part["color_group"].isin(HELDOUT_COLORS)]
    seen = part[part["color_group"].isin(SEEN_COLORS)]
    photo_rows.append({
        "variant": "photometric_aug",
        "model_seed": seed,
        "mean_dice": part["target_dice"].mean(),
        "seen_color_dice": seen["target_dice"].mean(),
        "heldout_color_dice": heldout["target_dice"].mean(),
        "red_dice": part[part["color_group"] == "red"]["target_dice"].mean(),
        "purple_dice": part[part["color_group"] == "purple"]["target_dice"].mean(),
        "worst_combo_dice": part.groupby(["color_group", "defect_type"])["target_dice"].mean().min(),
    })

all_seed_metrics = pd.concat([pd.DataFrame(baseline_rows), pd.DataFrame(photo_rows), adapter_seed_metrics], ignore_index=True)
all_seed_metrics.to_csv(out_dir / "color_shortcut_all_seed_metrics.csv", index=False, encoding="utf-8-sig")
summary = (
    all_seed_metrics.groupby("variant")[["mean_dice", "seen_color_dice", "heldout_color_dice", "red_dice", "purple_dice", "worst_combo_dice"]]
    .agg(["mean", "std", "count"])
    .reset_index()
)
summary.columns = ["_".join([x for x in col if x]) if isinstance(col, tuple) else col for col in summary.columns]
summary.to_csv(out_dir / "color_shortcut_comparison_summary.csv", index=False, encoding="utf-8-sig")
display(all_seed_metrics)
display(summary)

,variant,model_seed,mean_dice,seen_color_dice,heldout_color_dice,red_dice,purple_dice,worst_combo_dice,run_dir
0,baseline_no_aug,0,0.488118,0.654780,0.238125,0.000000,0.476249,0.000000,NaN
1,baseline_no_aug,1,0.548673,0.664565,0.374835,0.102099,0.647571,0.000000,NaN
2,baseline_no_aug,2,0.557193,0.700636,0.342029,0.069435,0.614624,0.000000,NaN
3,photometric_aug,0,0.595021,0.626223,0.548219,0.421596,0.674842,0.223290,NaN
4,photometric_aug,1,0.501883,0.558439,0.417048,0.271899,0.562197,0.014388,NaN
5,photometric_aug,2,0.657476,0.673091,0.634054,0.584392,0.683715,0.287654,NaN
6,adapter_identity_ce,0,0.567455,0.694609,0.376725,0.164099,0.589351,0.004307,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
7,adapter_identity_ce,1,0.613430,0.681135,0.511873,0.399161,0.624584,0.278809,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
8,adapter_identity_ce,2,0.587336,0.740827,0.357099,0.074927,0.639272,0.000000,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
9,adapter_identity_consistency,0,0.708458,0.739674,0.661634,0.593135,0.730132,0.355221,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...


,variant,mean_dice_mean,mean_dice_std,mean_dice_count,seen_color_dice_mean,seen_color_dice_std,seen_color_dice_count,heldout_color_dice_mean,heldout_color_dice_std,heldout_color_dice_count,red_dice_mean,red_dice_std,red_dice_count,purple_dice_mean,purple_dice_std,purple_dice_count,worst_combo_dice_mean,worst_combo_dice_std,worst_combo_dice_count
0,adapter_identity_ce,0.589407,0.023057,3,0.705523,0.031307,3,0.415232,0.084266,3,0.212729,0.167498,3,0.617735,0.025655,3,0.094372,0.159742,3
1,adapter_identity_consistency,0.629955,0.155879,3,0.667929,0.130999,3,0.572994,0.193564,3,0.461672,0.281637,3,0.684317,0.105526,3,0.269341,0.199144,3
2,baseline_no_aug,0.531328,0.037663,3,0.673327,0.024151,3,0.318329,0.071370,3,0.057178,0.052141,3,0.579481,0.090906,3,0.000000,0.000000,3
3,photometric_aug,0.584793,0.078299,3,0.619251,0.057643,3,0.533107,0.109289,3,0.425962,0.156292,3,0.640252,0.067743,3,0.175111,0.142862,3


## 38-5. Counterfactual 안정성 평가

In [6]:
cf_rows = []
for variant in ["adapter_identity_ce", "adapter_identity_consistency"]:
    run_dir = run_root / variant / "seed_0"
    model, device = load_pretrained_color_adapter_model(run_dir)
    cf_dir = out_dir / "counterfactual" / variant / "seed_0"
    cf = evaluate_counterfactual_with_loaded_model(
        model,
        device,
        manifests["eval_color_counterfactual_probe"],
        cf_dir,
        transforms=DEFAULT_COUNTERFACTUALS,
        max_samples=None,
        seed=38,
    )
    cf["variant"] = variant
    cf_rows.append(cf)

cf_all = pd.concat(cf_rows, ignore_index=True)
cf_all.to_csv(out_dir / "color_adapter_counterfactual_metrics.csv", index=False, encoding="utf-8-sig")
cf_summary = (
    cf_all.groupby(["variant", "transform", "color_group"])[["target_dice", "target_fnr", "prediction_flip_rate"]]
    .mean()
    .reset_index()
)
cf_summary.to_csv(out_dir / "color_adapter_counterfactual_summary.csv", index=False, encoding="utf-8-sig")
display(cf_summary[(cf_summary["color_group"] == "red") & (cf_summary["transform"].isin(["original", "grayscale", "gray_world", "channel_shuffle_bgr"]))])

,variant,transform,color_group,target_dice,target_fnr,prediction_flip_rate
11,adapter_identity_ce,channel_shuffle_bgr,red,0.599325,0.449855,0.025932
23,adapter_identity_ce,gray_world,red,0.546571,0.495476,0.026174
27,adapter_identity_ce,grayscale,red,0.490792,0.547112,0.022261
35,adapter_identity_ce,original,red,0.164478,0.858347,0.000000
51,adapter_identity_consistency,channel_shuffle_bgr,red,0.613869,0.361641,0.006049
63,adapter_identity_consistency,gray_world,red,0.616301,0.370351,0.008667
67,adapter_identity_consistency,grayscale,red,0.628774,0.359936,0.006565
75,adapter_identity_consistency,original,red,0.593124,0.400496,0.000000


## 38-6. 판정 리포트 작성

In [7]:
summary = pd.read_csv(out_dir / "color_shortcut_comparison_summary.csv")
seed_metrics = pd.read_csv(out_dir / "color_shortcut_all_seed_metrics.csv")
cf_summary = pd.read_csv(out_dir / "color_adapter_counterfactual_summary.csv")

def metric(variant, column):
    row = summary[summary["variant"] == variant].iloc[0]
    return float(row[f"{column}_mean"])

baseline_red = metric("baseline_no_aug", "red_dice")
baseline_seen = metric("baseline_no_aug", "seen_color_dice")
lines = [
    "# Pretrained Color-Invariant Adapter 실험 리포트",
    "",
    "## 1. 실험 조건",
    "- SegFormer-B0 본체는 pretrained 조건을 유지했다.",
    "- adapter는 RGB identity로 초기화했다. 따라서 초기 forward는 기존 RGB 입력과 같다.",
    "- train/eval manifest, seed, epoch, batch size, lr은 Chapter 2-2 baseline과 맞췄다.",
    "- from-scratch 모델과 pretrained 모델을 직접 비교하지 않았다.",
    "",
    "## 2. 주요 성능",
    "",
    "| variant | red Dice | heldout Dice | seen Dice | worst combo |",
    "|---|---:|---:|---:|---:|",
]
for variant in summary["variant"].tolist():
    lines.append(
        f"| {variant} | {metric(variant, 'red_dice'):.3f} | {metric(variant, 'heldout_color_dice'):.3f} | {metric(variant, 'seen_color_dice'):.3f} | {metric(variant, 'worst_combo_dice'):.3f} |"
    )

lines.extend(["", "## 3. 판정", ""])
for variant in ["adapter_identity_ce", "adapter_identity_consistency"]:
    if variant not in summary["variant"].values:
        continue
    red_delta = metric(variant, "red_dice") - baseline_red
    seen_delta = metric(variant, "seen_color_dice") - baseline_seen
    lines.append(f"- {variant}: red Dice delta={red_delta:.3f}, seen Dice delta={seen_delta:.3f}")

red_cf = cf_summary[(cf_summary["color_group"] == "red") & (cf_summary["transform"].isin(["original", "grayscale", "gray_world"]))]
lines.append("")
lines.append("## 4. Red counterfactual")
lines.append("```text")
lines.append(red_cf.to_string(index=False))
lines.append("```")
lines.append("")
lines.append("## 5. 해석")
lines.append("- red Dice가 오르면서 seen Dice 하락이 작으면 색 shortcut 완화로 해석할 수 있다.")
lines.append("- red original과 grayscale/gray_world 간 gap이 줄어들면 색상 counterfactual 안정성이 개선된 것이다.")
lines.append("- photometric_aug와 비교해 추가 이득이 없으면 구조 adapter보다 augmentation이 더 효율적인 접근일 수 있다.")

report_path = out_dir / "pretrained_color_adapter_report.md"
report_path.write_text("\n".join(lines), encoding="utf-8")
print(report_path)
print("\n".join(lines))

C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\color_adapter_pretrained\pretrained_color_adapter_report.md
# Pretrained Color-Invariant Adapter 실험 리포트

## 1. 실험 조건
- SegFormer-B0 본체는 pretrained 조건을 유지했다.
- adapter는 RGB identity로 초기화했다. 따라서 초기 forward는 기존 RGB 입력과 같다.
- train/eval manifest, seed, epoch, batch size, lr은 Chapter 2-2 baseline과 맞췄다.
- from-scratch 모델과 pretrained 모델을 직접 비교하지 않았다.

## 2. 주요 성능

| variant | red Dice | heldout Dice | seen Dice | worst combo |
|---|---:|---:|---:|---:|
| adapter_identity_ce | 0.213 | 0.415 | 0.706 | 0.094 |
| adapter_identity_consistency | 0.462 | 0.573 | 0.668 | 0.269 |
| baseline_no_aug | 0.057 | 0.318 | 0.673 | 0.000 |
| photometric_aug | 0.426 | 0.533 | 0.619 | 0.175 |

## 3. 판정

- adapter_identity_ce: red Dice delta=0.156, seen Dice delta=0.032
- adapter_identity_consistency: red Dice delta=0.404, seen Dice delta=-0.005

## 4. Red counterfactual
```text
                     variant  transform color_group  target_dice  target